# TweetSense Sentiment Comparison

This notebook compares two sentiment-classification methods:

1. **Traditional ML** — text preprocessing, TF-IDF vectorization, and classic classifiers.
2. **Hugging Face Zero-Shot** — a pre-trained transformer model (no fine-tuning needed).

Labels: `Positive`, `Negative`, `Neutral`, `Irrelevant`.

## 1. Imports and Settings

In [ ]:
import re
import string
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

# --- Paths ---
TRAIN_PATH = r"C:\Users\nwf15\OneDrive\Desktop\NLP_Project\twitter_training.csv"
VALIDATION_PATH = r"C:\Users\nwf15\OneDrive\Desktop\NLP_Project\twitter_validation.csv"

LABELS = ["Positive", "Negative", "Neutral", "Irrelevant"]
RANDOM_STATE = 42
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")

print("Training file exists:", Path(TRAIN_PATH).exists())
print("Validation file exists:", Path(VALIDATION_PATH).exists())

## 2. NLTK Setup

In [ ]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
lemmatizer = WordNetLemmatizer()
print("NLTK ready.")

## 3. Load and Clean Data

In [ ]:
columns = ["tweet_id", "entity", "sentiment", "tweet"]

train_df = pd.read_csv(TRAIN_PATH, header=None, names=columns)
valid_df = pd.read_csv(VALIDATION_PATH, header=None, names=columns)

# Drop missing values and keep only known labels
for df in [train_df, valid_df]:
    df.dropna(subset=["tweet", "sentiment"], inplace=True)

train_df = train_df[train_df["sentiment"].isin(LABELS)].reset_index(drop=True)
valid_df = valid_df[valid_df["sentiment"].isin(LABELS)].reset_index(drop=True)

print(f"Training: {train_df.shape[0]:,} rows")
print(f"Validation: {valid_df.shape[0]:,} rows")
display(train_df.head())
display(train_df["sentiment"].value_counts())

## 4. Text Preprocessing

In [ ]:
def clean_tweet(text):
    """Clean a tweet: remove URLs, mentions, hashtag symbols, punctuation, stopwords, then lemmatize."""
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"@\w+", " ", text)                     # mentions
    text = re.sub(r"#(\w+)", r"\1", text)                  # hashtag symbols
    text = text.encode("ascii", "ignore").decode("ascii")   # non-ASCII
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = tokenizer.tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w.isalpha() and w not in stop_words]
    return " ".join(tokens)


# Quick demo
example = "I LOVE this update!!! https://example.com #Amazing @user"
print("Original:", example)
print("Cleaned: ", clean_tweet(example))

## 5. Preprocess Datasets

In [ ]:
train_df["clean"] = train_df["tweet"].apply(clean_tweet)
valid_df["clean"] = valid_df["tweet"].apply(clean_tweet)

# Remove rows where cleaning produced empty text
train_df = train_df[train_df["clean"].str.strip().ne("")].reset_index(drop=True)
valid_df = valid_df[valid_df["clean"].str.strip().ne("")].reset_index(drop=True)

print(f"Training rows after cleaning: {len(train_df):,}")
print(f"Validation rows after cleaning: {len(valid_df):,}")
train_df[["tweet", "clean"]].head()

## 6. Train Traditional ML Models

Four classifiers are compared using optimized TF-IDF features (uni/bi/trigrams with sublinear TF scaling).

In [ ]:
X_train, y_train = train_df["clean"], train_df["sentiment"]
X_valid, y_valid = valid_df["clean"], valid_df["sentiment"]

# Classifiers with tuned hyperparameters
classifiers = {
    "Logistic Regression": LogisticRegression(C=5.0, max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE),
    "Linear SVM":         LinearSVC(C=1.0, dual="auto", max_iter=2000, random_state=RANDOM_STATE),
    "Naive Bayes":        MultinomialNB(alpha=0.1),
    "Random Forest":      RandomForestClassifier(n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1),
}

results = []
saved_models = {}

for name, clf in classifiers.items():
    print(f"Training: {name} ...", end=" ")

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_df=0.95,
                                  sublinear_tf=True, max_features=100_000)),
        ("clf", clf),
    ])

    t0 = time.time()
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_valid)
    runtime = time.time() - t0

    acc = accuracy_score(y_valid, preds)
    mf1 = f1_score(y_valid, preds, labels=LABELS, average="macro")
    wf1 = f1_score(y_valid, preds, labels=LABELS, average="weighted")

    results.append({"model": name, "accuracy": acc, "macro_f1": mf1, "weighted_f1": wf1, "seconds": round(runtime, 1)})
    saved_models[name] = (pipe, preds)
    print(f"Accuracy: {acc:.4f}  Macro-F1: {mf1:.4f}  ({runtime:.1f}s)")

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_df

## 7. Best Model — Detailed Report & Confusion Matrix

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model, best_preds = saved_models[best_name]

print(f"Best model: {best_name}\n")
print(classification_report(y_valid, best_preds, labels=LABELS))

# Confusion matrix
matrix = confusion_matrix(y_valid, best_preds, labels=LABELS)
plt.figure(figsize=(7, 5))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Save model
model_path = MODEL_DIR / "tweetsense_tfidf_model.joblib"
joblib.dump({"model": best_model, "labels": LABELS}, model_path)
print("Saved best model to:", model_path)

## 8. Hugging Face Zero-Shot Classification

Uses a pre-trained transformer (`facebook/bart-large-mnli`) for zero-shot classification on a small balanced sample. No fine-tuning required — the model classifies based on natural language understanding.

In [ ]:
from transformers import pipeline

HF_MODEL_NAME = "facebook/bart-large-mnli"
HF_SAMPLE_PER_CLASS = 5

hf_sample = (
    valid_df
    .groupby("sentiment", group_keys=False)
    .apply(lambda group: group.sample(min(len(group), HF_SAMPLE_PER_CLASS), random_state=RANDOM_STATE))
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(f"Hugging Face sample size: {len(hf_sample)}")
hf_sample[["entity", "sentiment", "tweet"]].head()

In [ ]:
zero_shot_clf = pipeline("zero-shot-classification", model=HF_MODEL_NAME, device=-1)

hf_preds, hf_scores = [], []
t0 = time.time()

for _, row in hf_sample.iterrows():
    result = zero_shot_clf(
        f"Tweet about {row['entity']}: {row['tweet']}",
        candidate_labels=LABELS,
        hypothesis_template="This tweet sentiment is {}.",
    )
    hf_preds.append(result["labels"][0])
    hf_scores.append(result["scores"][0])

hf_runtime = time.time() - t0
hf_sample["hf_prediction"] = hf_preds
hf_sample["hf_score"] = hf_scores

print(classification_report(hf_sample["sentiment"], hf_sample["hf_prediction"], labels=LABELS))
hf_sample[["entity", "tweet", "sentiment", "hf_prediction", "hf_score"]].head(10)

## 9. Final Comparison

In [ ]:
best_row = results_df.iloc[0].to_dict()

hf_metrics = {
    "model": HF_MODEL_NAME,
    "accuracy": accuracy_score(hf_sample["sentiment"], hf_sample["hf_prediction"]),
    "macro_f1": f1_score(hf_sample["sentiment"], hf_sample["hf_prediction"], labels=LABELS, average="macro"),
    "weighted_f1": f1_score(hf_sample["sentiment"], hf_sample["hf_prediction"], labels=LABELS, average="weighted"),
    "seconds": round(hf_runtime, 1),
}

comparison_df = pd.DataFrame([best_row, hf_metrics])
comparison_df.index = ["Best Traditional (TF-IDF)", "Hugging Face Zero-Shot"]
comparison_df[["model", "accuracy", "macro_f1", "weighted_f1", "seconds"]]

## 10. Try the Saved Model

In [ ]:
loaded = joblib.load(model_path)
loaded_model = loaded["model"]

new_tweets = [
    "I love this game. The new update is amazing!",
    "This service is terrible and keeps crashing.",
    "The product page was updated today.",
    "I made lunch while everyone was talking about the match.",
]

clean_new = [clean_tweet(t) for t in new_tweets]
predictions = loaded_model.predict(clean_new)

pd.DataFrame({"tweet": new_tweets, "prediction": predictions})